In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Importing Libs

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Utilizing GPU

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


## Dataset Class

In [4]:
class AnimalDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

        # get unique labels and create mapping
        self.labels = sorted(self.data['label'].unique())
        self.label_to_idx = {label: idx for idx, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label']

        # load image
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label_idx = self.label_to_idx[label]
        return image, label_idx

## Transforms

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Evalutaion Function

In [6]:
def evaluate_model(model, data_loader, return_predictions=True):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    if return_predictions:
        return all_preds, all_labels
    else:
        acc = accuracy_score(all_labels, all_preds)
        return acc * 100

## Training Function

In [7]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix({'loss': running_loss/len(train_loader), 'acc': 100*correct/total})

        train_acc = 100 * correct / total

        # validation
        val_acc = evaluate_model(model, val_loader, return_predictions=False)
        print(f'Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))
    return model

## Calculate Metrics

In [8]:
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return {
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

## Training

In [9]:
base_path = '/content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv'
num_folds = 5

results = {
    'ResNet50': [],

}

for fold in range(1, num_folds + 1):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}/{num_folds}')
    print(f'{"="*60}')

    fold_path = os.path.join(base_path, f'fold_{fold}')

    train_csv = os.path.join(fold_path, 'train.csv')
    val_csv = os.path.join(fold_path, 'val.csv')
    test_csv = os.path.join(fold_path, 'test.csv')

    # create datasets
    train_dataset = AnimalDataset(train_csv, transform=train_transform)
    val_dataset = AnimalDataset(val_csv, transform=test_transform)
    test_dataset = AnimalDataset(test_csv, transform=test_transform)

    num_classes = len(train_dataset.labels)
    print(f'Number of classes: {num_classes}')
    print(f'Classes: {train_dataset.labels}')

    # create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

    # train ResNet50
    print(f'\nTraining ResNet50 on Fold {fold}...')
    resnet50_model = models.resnet50(pretrained=True)

    # Freeze all parameters in the network
    # for param in resnet50_model.parameters():
    #     param.requires_grad = False

    # Get the number of input features for the final fully connected layer
    num_ftrs = resnet50_model.fc.in_features
    # Replace the final fully connected layer
    resnet50_model.fc = nn.Linear(num_ftrs, num_classes)

    resnet50_model = resnet50_model.to(device)

    # Only parameters that have requires_grad=True (the new fc layer) will be optimized
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(resnet50_model.parameters(), lr=0.0001)

    resnet50_model = train_model(resnet50_model, train_loader, val_loader, criterion, optimizer, epochs=5)

    preds, labels = evaluate_model(resnet50_model, test_loader)
    resnet50_metrics = calculate_metrics(labels, preds)
    results['ResNet50'].append(resnet50_metrics)

    print(f'\nResNet50 Results (Fold {fold}):')
    for metric, value in resnet50_metrics.items():
        print(f'{metric}: {value:.4f}')


FOLD 1/5
Number of classes: 64
Classes: ['antelope', 'bear', 'beaver', 'bee', 'bison', 'blackbird', 'buffalo', 'butterfly', 'camel', 'cat', 'cheetah', 'chimpanzee', 'chinchilla', 'cow', 'crab', 'crocodile', 'deer', 'dog', 'dolphin', 'donkey', 'duck', 'eagle', 'elephant', 'falcon', 'ferret', 'flamingo', 'fox', 'frog', 'giraffe', 'goat', 'goose', 'gorilla', 'grasshopper', 'hawk', 'hedgehog', 'hippopotamus', 'hyena', 'iguana', 'jaguar', 'kangaroo', 'koala', 'lemur', 'leopard', 'lizard', 'lynx', 'mole', 'mongoose', 'ostrich', 'otter', 'owl', 'panda', 'peacock', 'penguin', 'porcupine', 'raccoon', 'seal', 'sheep', 'snail', 'snake', 'spider', 'squid', 'walrus', 'whale', 'wolf']

Training ResNet50 on Fold 1...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s]
Epoch 1/5: 100%|██████████| 288/288 [1:10:10<00:00, 14.62s/it, loss=0.497, acc=92.6]


Epoch 1: Train Acc = 92.63%, Val Acc = 99.78%


Epoch 2/5: 100%|██████████| 288/288 [03:02<00:00,  1.58it/s, loss=0.0396, acc=99.2]


Epoch 2: Train Acc = 99.23%, Val Acc = 99.17%


...

Epoch 4: Train Acc = 99.41%, Val Acc = 99.44%


Epoch 5/5: 100%|██████████| 288/288 [02:54<00:00,  1.65it/s, loss=0.033, acc=99.3]


Epoch 5: Train Acc = 99.28%, Val Acc = 98.22%

ResNet50 Results (Fold 1):
Accuracy: 0.9986
Precision: 0.9987
Recall: 0.9986
F1-Score: 0.9986

FOLD 2/5
Number of classes: 64
Classes: ['antelope', 'bear', 'beaver', 'bee', 'bison', 'blackbird', 'buffalo', 'butterfly', 'camel', 'cat', 'cheetah', 'chimpanzee', 'chinchilla', 'cow', 'crab', 'crocodile', 'deer', 'dog', 'dolphin', 'donkey', 'duck', 'eagle', 'elephant', 'falcon', 'ferret', 'flamingo', 'fox', 'frog', 'giraffe', 'goat', 'goose', 'gorilla', 'grasshopper', 'hawk', 'hedgehog', 'hippopotamus', 'hyena', 'iguana', 'jaguar', 'kangaroo', 'koala', 'lemur', 'leopard', 'lizard', 'lynx', 'mole', 'mongoose', 'ostrich', 'otter', 'owl', 'panda', 'peacock', 'penguin', 'porcupine', 'raccoon', 'seal', 'sheep', 'snail', 'snake', 'spider', 'squid', 'walrus', 'whale', 'wolf']

Training ResNet50 on Fold 2...


Epoch 1/5:  72%|███████▏  | 206/288 [02:08<00:51,  1.60it/s, loss=0.481, acc=90.8]


KeyboardInterrupt: 

## Results

In [ ]:
print('\n' + '='*70)
print('FINAL RESULTS ACROSS ALL FOLDS')
print('='*70)

model_name = 'GoogLeNet' # Corrected: use the string key 'GoogLeNet'
print(f'\n{model_name}:')
print('-'*70)

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for metric in metrics_names:
      values = [fold[metric] for fold in results[model_name]]
      mean_val = np.mean(values)
      std_val = np.std(values)
      print(f'{metric:12s}: {mean_val:.4f} \u00b1 {std_val:.4f}')